In [1]:
#Cell 1 — load and inspect (raw, no cleaning yet):
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from src.utils import load_config

cfg = load_config()

path_2018 = os.path.join(cfg["paths"]["cicids2018_dir"], "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv")

df_2018_raw = pd.read_csv(path_2018, low_memory=False)
df_2018_raw.columns = [str(c).strip() for c in df_2018_raw.columns]

print("Shape:", df_2018_raw.shape)
print("\nColumn names:")
for i, c in enumerate(df_2018_raw.columns, 1):
    print(f"{i:>3}. {c}")

Shape: (1048575, 80)

Column names:
  1. Dst Port
  2. Protocol
  3. Timestamp
  4. Flow Duration
  5. Tot Fwd Pkts
  6. Tot Bwd Pkts
  7. TotLen Fwd Pkts
  8. TotLen Bwd Pkts
  9. Fwd Pkt Len Max
 10. Fwd Pkt Len Min
 11. Fwd Pkt Len Mean
 12. Fwd Pkt Len Std
 13. Bwd Pkt Len Max
 14. Bwd Pkt Len Min
 15. Bwd Pkt Len Mean
 16. Bwd Pkt Len Std
 17. Flow Byts/s
 18. Flow Pkts/s
 19. Flow IAT Mean
 20. Flow IAT Std
 21. Flow IAT Max
 22. Flow IAT Min
 23. Fwd IAT Tot
 24. Fwd IAT Mean
 25. Fwd IAT Std
 26. Fwd IAT Max
 27. Fwd IAT Min
 28. Bwd IAT Tot
 29. Bwd IAT Mean
 30. Bwd IAT Std
 31. Bwd IAT Max
 32. Bwd IAT Min
 33. Fwd PSH Flags
 34. Bwd PSH Flags
 35. Fwd URG Flags
 36. Bwd URG Flags
 37. Fwd Header Len
 38. Bwd Header Len
 39. Fwd Pkts/s
 40. Bwd Pkts/s
 41. Pkt Len Min
 42. Pkt Len Max
 43. Pkt Len Mean
 44. Pkt Len Std
 45. Pkt Len Var
 46. FIN Flag Cnt
 47. SYN Flag Cnt
 48. RST Flag Cnt
 49. PSH Flag Cnt
 50. ACK Flag Cnt
 51. URG Flag Cnt
 52. CWE Flag Count
 53. ECE Flag

In [2]:
#Cell 2 — check the label column:
# The label column in 2018 files is sometimes named 'Label' (same as 2017)
# but let's confirm, and see the exact class names/text used
label_candidates = [c for c in df_2018_raw.columns if "label" in c.lower()]
print("Label-like columns found:", label_candidates)

label_col_2018 = label_candidates[0]
print(f"\nUsing label column: '{label_col_2018}'")
print("\nClass distribution:")
print(df_2018_raw[label_col_2018].value_counts())

Label-like columns found: ['Label']

Using label column: 'Label'

Class distribution:
Label
Benign    762384
Bot       286191
Name: count, dtype: int64


In [3]:
#Cell 3 — compare column sets between 2017 and 2018:
# Load our cleaned 2017 columns for comparison (before the zero-variance drop,
# to compare apples-to-apples against 2018's raw column set)
cols_2017_raw = set([
    "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Fwd Packet Length Max",
    "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
    "Bwd Packet Length Std", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean",
    "Flow IAT Std", "Flow IAT Max", "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean",
    "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean",
    "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags", "Fwd Header Length", "Bwd Header Length",
    "Fwd Packets/s", "Bwd Packets/s", "Min Packet Length", "Max Packet Length",
    "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count",
    "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size",
    "Avg Bwd Segment Size", "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
    "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
    "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward",
    "Active Mean", "Active Std", "Active Max", "Active Min", "Idle Mean",
    "Idle Std", "Idle Max", "Idle Min", "Label"
])

cols_2018 = set(df_2018_raw.columns)

common = cols_2017_raw & cols_2018
only_2017 = cols_2017_raw - cols_2018
only_2018 = cols_2018 - cols_2017_raw

print(f"2017 columns:        {len(cols_2017_raw)}")
print(f"2018 columns:        {len(cols_2018)}")
print(f"Common columns:      {len(common)}")
print(f"\nOnly in 2017 ({len(only_2017)}):")
print(sorted(only_2017))
print(f"\nOnly in 2018 ({len(only_2018)}):")
print(sorted(only_2018))

2017 columns:        78
2018 columns:        80
Common columns:      28

Only in 2017 (50):
['ACK Flag Count', 'Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Header Length', 'Bwd IAT Total', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Bwd Packets/s', 'Destination Port', 'ECE Flag Count', 'FIN Flag Count', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Header Length', 'Fwd IAT Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 'Fwd Packets/s', 'Init_Win_bytes_backward', 'Init_Win_bytes_forward', 'Max Packet Length', 'Min Packet Length', 'PSH Flag Count', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'RST Flag Count', 'SYN Flag Count', 'Subflow Bwd Bytes', 'Subflow Bwd Packets', 'Subflow Fwd Bytes',